**<h1>Electric Line Extension - Standardization 1**
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 04/22/2026 | Start Development: 04/22/2026</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for Q1(Revised) 2025 - Q3 2025
* Goals: 1. Update all column names and text across workbook. Remove inconsistencies in naming for master document / dashboard.


In [7]:
import openpyxl
import re

# Load workbook
file_path = r"C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed/2023&2024_Composite.4.22.xlsx"
wb = openpyxl.load_workbook(file_path)

In [8]:
# Target sheets
data_sheets = [
    "PG&E_2024", "SCE_2024", "PG&E_2023",
    "SDGE_2024", "SCE_2023", "SDGE_2023"
]

qc_sheet = wb["QC Check"]

# --- STANDARD TARGET VALUES ---
IOU_STANDARD = "Total Electric Line or Service Extension Applications (IOU Installed)"
APPLICANT_STANDARD = "Total Electric Line or Service Extension Applications (Applicant Installed)"

# --- NORMALIZATION FUNCTION ---
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize(text):
    return clean_text(text).lower()

# --- VALUE MAPPING (FOR "Type" COLUMN VALUES) ---
def map_type_value(val):
    n = normalize(val)

    # IOU Installed (Requests Received)
    if (
        "requests received" in n
        and "application" in n
    ):
        return IOU_STANDARD

    # Applicant Installed
    if "applicant install" in n:
        return APPLICANT_STANDARD

    return clean_text(val)

In [9]:
# --- STEP 1: FIX "Type" COLUMN VALUES ---
for sheet_name in data_sheets:
    ws = wb[sheet_name]

    # Find "Type" column index dynamically
    headers = [cell.value for cell in ws[1]]
    try:
        type_col_idx = headers.index("Type") + 1
    except ValueError:
        print(f"'Type' column not found in {sheet_name}")
        continue

    for row in ws.iter_rows(min_row=2):
        cell = row[type_col_idx - 1]
        original = cell.value
        new_val = map_type_value(original)

        if original != new_val:
            cell.value = new_val


In [10]:
# --- STEP 2: FIX QC FORMULAS ---
def fix_formula(formula):
    if not isinstance(formula, str) or not formula.startswith("="):
        return formula

    # Detect if it's your SUM(SUMIFS(...)) pattern
    if "SUMIFS" not in formula:
        return formula

    # Replace the ENTIRE array constant safely
    # Works for both comma and semicolon locales
    formula = re.sub(
        r'\{\s*"Total Electric Line Extension Requests.*?"\s*[,;]\s*"Total Electric Line Extension Applications.*?"\s*\}',
        '{"Total Electric Line or Service Extension Applications (IOU Installed)";'
        '"Total Electric Line or Service Extension Applications (Applicant Installed)"}',
        formula,
        flags=re.IGNORECASE
    )

    return formula

for row in qc_sheet.iter_rows():
    for cell in row:
        if isinstance(cell.value, str) and cell.value.startswith("="):
            new_formula = fix_formula(cell.value)
            if new_formula != cell.value:
                print(f"Updated formula: {cell.coordinate}")
                cell.value = new_formula


In [11]:
# --- SAVE ---
output_file = r"2023&2024_Composite_FIXED.xlsx"
wb.save(output_file)

print("DONE →", output_file)

DONE → 2023&2024_Composite_FIXED.xlsx
